<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Activity_%E2%80%93_Supplier_Risk_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# SUPPLIER RISK & PERFORMANCE ANALYSIS
# ================================================================
#
# OBJETIVO DEL EJERCICIO
#
# Analizar diferentes proveedores considerando simultáneamente:
#
# - Cost
# - Lead Time
# - Delivery Performance
# - Quality
# - Supplier Dependency
# - Geographic Risk
# - Financial Risk
# - Transportation Risk
#
# El propósito NO es simplemente identificar al proveedor
# más barato, sino entender:
#
# 1. ¿Qué proveedores tienen buen desempeño?
# 2. ¿Qué proveedores representan mayor riesgo?
# 3. ¿Qué proveedores requieren atención inmediata?
# 4. ¿Qué estrategia debería adoptar la empresa?
#
# Estrategias posibles:
#
# MAINTAIN   = mantener
# DEVELOP    = desarrollar/mejorar
# DIVERSIFY  = reducir dependencia / buscar segunda fuente
# REPLACE    = considerar reemplazo
#
# ================================================================


# ================================================================
# 1. IMPORTAR LIBRERÍAS
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ================================================================
# 2. CREAR BASE DE DATOS DE PROVEEDORES
# ================================================================
#
# Para que los estudiantes puedan ejecutar el ejercicio
# sin necesidad de cargar archivos externos,
# vamos a crear una base ficticia.
#
# Más adelante esta misma estructura podría reemplazarse
# por una base real en Excel.
#
# ================================================================


data = {

    "Supplier": [
        "Alpha Global",
        "Beta Industries",
        "Gamma Supply",
        "Delta Strategic",
        "Epsilon Parts",
        "Zeta Manufacturing",
        "Omega International",
        "Nova Components"
    ],

    # ------------------------------------------------------------
    # COST
    #
    # Índice relativo de costo.
    #
    # Valores menores = mejor desempeño.
    #
    # ------------------------------------------------------------

    "Cost_Index": [
        88,
        95,
        82,
        105,
        91,
        86,
        102,
        89
    ],

    # ------------------------------------------------------------
    # LEAD TIME
    #
    # Tiempo promedio de entrega en días.
    #
    # Menor Lead Time normalmente significa
    # menor exposición a incertidumbre.
    #
    # ------------------------------------------------------------

    "Lead_Time_Days": [
        22,
        14,
        38,
        12,
        27,
        35,
        18,
        25
    ],

    # ------------------------------------------------------------
    # ON-TIME DELIVERY
    #
    # Porcentaje de entregas realizadas a tiempo.
    #
    # Mayor porcentaje = mejor desempeño.
    #
    # ------------------------------------------------------------

    "On_Time_Delivery": [
        94,
        97,
        82,
        99,
        91,
        85,
        95,
        92
    ],

    # ------------------------------------------------------------
    # QUALITY
    #
    # Porcentaje de unidades aceptadas sin problemas.
    #
    # Mayor porcentaje = mejor.
    #
    # ------------------------------------------------------------

    "Quality_Performance": [
        96,
        98,
        88,
        99,
        94,
        90,
        97,
        95
    ],

    # ------------------------------------------------------------
    # DEPENDENCY
    #
    # Porcentaje de nuestras compras que depende
    # de ese proveedor.
    #
    # Un porcentaje elevado aumenta el riesgo.
    #
    # Ejemplo:
    #
    # 70% significa que gran parte del abastecimiento
    # depende de un solo proveedor.
    #
    # ------------------------------------------------------------

    "Dependency_Percent": [
        35,
        20,
        65,
        25,
        45,
        70,
        30,
        40
    ],

    # ------------------------------------------------------------
    # GEOGRAPHIC RISK
    #
    # Escala de 1 a 10.
    #
    # 1 = muy bajo riesgo
    # 10 = riesgo geográfico muy alto
    #
    # Puede representar:
    #
    # - conflictos
    # - desastres naturales
    # - concentración regional
    # - infraestructura limitada
    #
    # ------------------------------------------------------------

    "Geographic_Risk": [
        3,
        2,
        8,
        2,
        5,
        7,
        4,
        5
    ],

    # ------------------------------------------------------------
    # FINANCIAL RISK
    #
    # Escala de 1 a 10.
    #
    # Un valor alto representa mayor vulnerabilidad
    # financiera del proveedor.
    #
    # ------------------------------------------------------------

    "Financial_Risk": [
        3,
        2,
        7,
        2,
        5,
        8,
        3,
        4
    ],

    # ------------------------------------------------------------
    # TRANSPORTATION RISK
    #
    # Escala de 1 a 10.
    #
    # Puede representar:
    #
    # - dependencia marítima
    # - rutas críticas
    # - congestión portuaria
    # - distancia
    # - vulnerabilidad logística
    #
    # ------------------------------------------------------------

    "Transportation_Risk": [
        4,
        2,
        9,
        2,
        6,
        8,
        5,
        5
    ]
}


df = pd.DataFrame(data)


print("\n" + "="*90)

print("ORIGINAL SUPPLIER DATA")

print("="*90)

print(df.to_string(index=False))


# ================================================================
# 3. NORMALIZACIÓN DE VARIABLES
# ================================================================
#
# Tenemos variables en escalas diferentes:
#
# Cost_Index = aproximadamente 80-110
#
# Lead_Time = días
#
# Delivery = porcentaje
#
# Risk = escala 1-10
#
# Para combinarlas necesitamos transformarlas
# a una escala comparable entre 0 y 100.
#
#
# UTILIZAREMOS MIN-MAX NORMALIZATION
#
# Fórmula:
#
# Normalized =
#
# (value - minimum)
# ------------------- x 100
# (maximum - minimum)
#
#
# De esta manera:
#
# 0   = mejor/mínimo valor
# 100 = peor/máximo valor
#
# ================================================================


def normalize_positive(series):

    """
    Esta función normaliza variables donde
    valores ALTOS significan algo BUENO.

    Ejemplos:

    On-Time Delivery
    Quality

    El mejor valor recibirá aproximadamente 100.
    """

    return (

        (series - series.min())

        /

        (series.max() - series.min())

        * 100

    )


def normalize_negative(series):

    """
    Esta función normaliza variables donde
    valores ALTOS significan algo MALO.

    Ejemplos:

    Cost
    Lead Time

    Aquí invertimos el resultado.

    El proveedor con menor costo o menor lead time
    recibe mayor puntuación.
    """

    return (

        100 -

        (
            (series - series.min())

            /

            (series.max() - series.min())

            * 100
        )

    )


# ================================================================
# 4. NORMALIZAR INDICADORES DE DESEMPEÑO
# ================================================================


df["Cost_Score"] = normalize_negative(
    df["Cost_Index"]
)


df["Lead_Time_Score"] = normalize_negative(
    df["Lead_Time_Days"]
)


df["Delivery_Score"] = normalize_positive(
    df["On_Time_Delivery"]
)


df["Quality_Score"] = normalize_positive(
    df["Quality_Performance"]
)


# ================================================================
# 5. CALCULAR SUPPLIER PERFORMANCE SCORE
# ================================================================
#
# Vamos a combinar cuatro criterios.
#
# Los pesos son una decisión gerencial.
#
# En este ejemplo:
#
# Cost            = 25%
# Lead Time       = 20%
# Delivery        = 30%
# Quality         = 25%
#
# Total           = 100%
#
#
# Los estudiantes pueden modificar estos pesos
# para analizar diferentes estrategias.
#
# ================================================================


df["Performance_Score"] = (

    df["Cost_Score"] * 0.25

    +

    df["Lead_Time_Score"] * 0.20

    +

    df["Delivery_Score"] * 0.30

    +

    df["Quality_Score"] * 0.25

)


# ================================================================
# 6. NORMALIZAR FACTORES DE RIESGO
# ================================================================
#
# Ahora construiremos el Supplier Risk Score.
#
# Aquí un valor ALTO siempre significa MÁS RIESGO.
#
# ================================================================


# ------------------------------------------------------------
# DEPENDENCY
#
# Ya está expresada en porcentaje.
#
# Podemos utilizarla directamente.
# ------------------------------------------------------------

df["Dependency_Risk"] = df["Dependency_Percent"]


# ------------------------------------------------------------
# OTROS RIESGOS
#
# Están en escala 1-10.
#
# Los llevamos a escala 0-100.
#
# ------------------------------------------------------------

df["Geo_Risk_Score"] = (

    df["Geographic_Risk"]
    / 10
    * 100

)


df["Financial_Risk_Score"] = (

    df["Financial_Risk"]
    / 10
    * 100

)


df["Transport_Risk_Score"] = (

    df["Transportation_Risk"]
    / 10
    * 100

)


# ================================================================
# 7. AGREGAR RIESGO OPERACIONAL DEL PROVEEDOR
# ================================================================
#
# Además de los factores externos,
# podemos incorporar desempeño deficiente.
#
# Por ejemplo:
#
# si Delivery Score es bajo,
# el riesgo operacional aumenta.
#
# Lo calculamos como:
#
# 100 - Delivery Score
#
# ================================================================


df["Delivery_Risk"] = (

    100 - df["Delivery_Score"]

)


# ================================================================
# 8. CALCULAR SUPPLIER RISK SCORE
# ================================================================
#
# Pesos utilizados:
#
# Dependency       = 25%
# Geographic Risk  = 20%
# Financial Risk   = 20%
# Transportation   = 20%
# Delivery Risk    = 15%
#
#
# IMPORTANTE:
#
# Estos pesos NO son universales.
#
# Son una decisión estratégica.
#
# Una empresa farmacéutica podría dar más peso
# a calidad y continuidad.
#
# Una empresa retail podría dar más peso
# a transporte y lead time.
#
# ================================================================


df["Risk_Score"] = (

    df["Dependency_Risk"] * 0.25

    +

    df["Geo_Risk_Score"] * 0.20

    +

    df["Financial_Risk_Score"] * 0.20

    +

    df["Transport_Risk_Score"] * 0.20

    +

    df["Delivery_Risk"] * 0.15

)


# ================================================================
# 9. CLASIFICAR NIVEL DE RIESGO
# ================================================================
#
# Crearemos tres categorías:
#
# LOW
# MEDIUM
# HIGH
#
# ================================================================


def risk_category(score):

    if score < 35:

        return "LOW"

    elif score < 60:

        return "MEDIUM"

    else:

        return "HIGH"


df["Risk_Level"] = (

    df["Risk_Score"]
    .apply(risk_category)

)


# ================================================================
# 10. CLASIFICAR NIVEL DE DESEMPEÑO
# ================================================================


def performance_category(score):

    if score >= 70:

        return "HIGH"

    elif score >= 45:

        return "MEDIUM"

    else:

        return "LOW"


df["Performance_Level"] = (

    df["Performance_Score"]
    .apply(performance_category)

)


# ================================================================
# 11. GENERAR RECOMENDACIÓN ESTRATÉGICA
# ================================================================
#
# La lógica será:
#
#
# HIGH PERFORMANCE + LOW RISK
#
#         MAINTAIN
#
#
# HIGH PERFORMANCE + HIGH RISK
#
#         DIVERSIFY
#
#
# MEDIUM PERFORMANCE
#
#         DEVELOP
#
#
# LOW PERFORMANCE + HIGH RISK
#
#         REPLACE
#
#
# ================================================================


def supplier_strategy(row):

    risk = row["Risk_Level"]

    performance = row["Performance_Level"]


    # --------------------------------------------------------
    # Excelente proveedor y bajo riesgo.
    # --------------------------------------------------------

    if performance == "HIGH" and risk == "LOW":

        return "MAINTAIN"


    # --------------------------------------------------------
    # Buen proveedor pero riesgo elevado.
    #
    # No necesariamente conviene eliminarlo.
    #
    # Puede ser mejor reducir dependencia.
    # --------------------------------------------------------

    elif performance == "HIGH" and risk in ["MEDIUM", "HIGH"]:

        return "DIVERSIFY"


    # --------------------------------------------------------
    # Desempeño intermedio.
    #
    # Puede existir oportunidad de desarrollo.
    # --------------------------------------------------------

    elif performance == "MEDIUM" and risk != "HIGH":

        return "DEVELOP"


    # --------------------------------------------------------
    # Desempeño bajo y riesgo alto.
    #
    # Es el escenario más crítico.
    # --------------------------------------------------------

    elif performance == "LOW" and risk == "HIGH":

        return "REPLACE"


    # --------------------------------------------------------
    # Otros casos.
    # --------------------------------------------------------

    elif performance == "LOW" and risk == "MEDIUM":

        return "DEVELOP"


    else:

        return "DIVERSIFY"


df["Recommended_Strategy"] = (

    df.apply(
        supplier_strategy,
        axis=1
    )

)


# ================================================================
# 12. MOSTRAR RESULTADOS GENERALES
# ================================================================


results = df[

    [
        "Supplier",
        "Performance_Score",
        "Risk_Score",
        "Performance_Level",
        "Risk_Level",
        "Recommended_Strategy"
    ]

].copy()


results = results.sort_values(

    "Risk_Score",

    ascending=False

)


print("\n" + "="*90)

print("SUPPLIER RISK & PERFORMANCE RESULTS")

print("="*90)


print(

    results
    .round(2)
    .to_string(index=False)

)


# ================================================================
# 13. GRÁFICA 1
#
# SUPPLIER RISK RANKING
# ================================================================


risk_plot = df.sort_values(
    "Risk_Score",
    ascending=True
)


# Asignaremos un color según riesgo.

risk_colors = []


for risk in risk_plot["Risk_Score"]:

    if risk >= 60:

        risk_colors.append("#E74C3C")     # ROJO

    elif risk >= 35:

        risk_colors.append("#F39C12")     # NARANJA

    else:

        risk_colors.append("#2ECC71")     # VERDE


plt.figure(
    figsize=(12,7)
)


bars = plt.barh(

    risk_plot["Supplier"],

    risk_plot["Risk_Score"],

    color=risk_colors

)


plt.title(

    "Supplier Risk Ranking",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Supplier Risk Score"
)


# Líneas que ayudan a interpretar categorías.

plt.axvline(

    35,

    color="#F1C40F",

    linestyle="--",

    alpha=0.8

)


plt.axvline(

    60,

    color="#E74C3C",

    linestyle="--",

    alpha=0.8

)


# Mostrar valor encima de cada barra.

for bar in bars:

    width = bar.get_width()

    plt.text(

        width + 1,

        bar.get_y()
        + bar.get_height()/2,

        f"{width:.1f}",

        va="center",

        fontweight="bold"

    )


plt.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 14. GRÁFICA 2
#
# RISK VS PERFORMANCE MAP
# ================================================================
#
# Esta es probablemente la gráfica
# MÁS IMPORTANTE DEL EJERCICIO.
#
#
# Eje X:
#
# Supplier Performance
#
#
# Eje Y:
#
# Supplier Risk
#
#
# Ideal:
#
# parte inferior derecha:
#
# HIGH PERFORMANCE
# LOW RISK
#
#
# Crítico:
#
# parte superior izquierda:
#
# LOW PERFORMANCE
# HIGH RISK
#
# ================================================================


plt.figure(
    figsize=(12,8)
)


strategy_colors = {

    "MAINTAIN": "#2ECC71",

    "DEVELOP": "#3498DB",

    "DIVERSIFY": "#F39C12",

    "REPLACE": "#E74C3C"

}


for strategy in [

    "MAINTAIN",
    "DEVELOP",
    "DIVERSIFY",
    "REPLACE"

]:

    subset = df[
        df["Recommended_Strategy"] == strategy
    ]


    plt.scatter(

        subset["Performance_Score"],

        subset["Risk_Score"],

        s=300,

        alpha=0.80,

        color=strategy_colors[strategy],

        label=strategy,

        edgecolor="black"

    )


# ------------------------------------------------------------
# Colocar el nombre del proveedor.
# ------------------------------------------------------------

for i, row in df.iterrows():

    plt.text(

        row["Performance_Score"] + 1,

        row["Risk_Score"] + 1,

        row["Supplier"],

        fontsize=9

    )


# ------------------------------------------------------------
# Líneas de referencia.
# ------------------------------------------------------------

plt.axhline(

    60,

    color="#E74C3C",

    linestyle="--",

    alpha=0.6

)


plt.axhline(

    35,

    color="#F39C12",

    linestyle="--",

    alpha=0.6

)


plt.axvline(

    70,

    color="#2ECC71",

    linestyle="--",

    alpha=0.6

)


plt.axvline(

    45,

    color="#3498DB",

    linestyle="--",

    alpha=0.6

)


plt.title(

    "Supplier Risk vs Performance Map",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(

    "Supplier Performance Score"

)


plt.ylabel(

    "Supplier Risk Score"

)


plt.legend(

    title="Recommended Strategy"

)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 15. GRÁFICA 3
#
# DEPENDENCY VS RISK
# ================================================================
#
# Esta gráfica ayuda a responder:
#
# ¿Estamos dependiendo demasiado
# de proveedores que además son riesgosos?
#
# ================================================================


plt.figure(
    figsize=(11,7)
)


plt.scatter(

    df["Dependency_Percent"],

    df["Risk_Score"],

    s=300,

    color="#9B59B6",

    alpha=0.75,

    edgecolor="black"

)


for i, row in df.iterrows():

    plt.text(

        row["Dependency_Percent"] + 1,

        row["Risk_Score"] + 1,

        row["Supplier"],

        fontsize=9

    )


plt.axvline(

    50,

    color="#E67E22",

    linestyle="--"

)


plt.axhline(

    60,

    color="#E74C3C",

    linestyle="--"

)


plt.title(

    "Supplier Dependency and Risk Exposure",

    fontsize=17,

    fontweight="bold"

)


plt.xlabel(

    "Supplier Dependency (%)"

)


plt.ylabel(

    "Supplier Risk Score"

)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 16. RADAR CHART
#
# COMPARACIÓN DE LOS PROVEEDORES
# ================================================================
#
# Crearemos un perfil multidimensional.
#
# En el radar queremos que:
#
# valores más altos = mejor situación.
#
# Por esto invertimos los factores de riesgo.
#
# ================================================================


radar_variables = [

    "Cost_Score",

    "Lead_Time_Score",

    "Delivery_Score",

    "Quality_Score"

]


radar_labels = [

    "Cost",

    "Lead Time",

    "Delivery",

    "Quality"

]


# Número de variables.

N = len(radar_variables)


angles = np.linspace(

    0,

    2 * np.pi,

    N,

    endpoint=False

).tolist()


angles += angles[:1]


supplier_colors = [

    "#E74C3C",
    "#3498DB",
    "#F39C12",
    "#2ECC71",
    "#9B59B6",
    "#1ABC9C",
    "#34495E",
    "#E67E22"

]


fig = plt.figure(
    figsize=(11,11)
)


ax = plt.subplot(
    111,
    polar=True
)


for index, row in df.iterrows():

    values = [

        row[var]

        for var in radar_variables

    ]


    values += values[:1]


    ax.plot(

        angles,

        values,

        linewidth=2,

        label=row["Supplier"],

        color=supplier_colors[index]

    )


    ax.fill(

        angles,

        values,

        color=supplier_colors[index],

        alpha=0.04

    )


ax.set_xticks(
    angles[:-1]
)


ax.set_xticklabels(
    radar_labels,
    fontsize=11
)


ax.set_title(

    "Supplier Performance Radar",

    fontsize=18,

    fontweight="bold",

    pad=25

)


ax.legend(

    bbox_to_anchor=(1.35, 1.10),

    loc="upper right"

)


plt.show()


# ================================================================
# 17. GRÁFICA 5
#
# MANAGEMENT ACTION MATRIX
# ================================================================
#
# Queremos saber cuántos proveedores
# quedan en cada estrategia.
#
# ================================================================


strategy_count = (

    df["Recommended_Strategy"]
    .value_counts()

)


strategy_order = [

    "MAINTAIN",
    "DEVELOP",
    "DIVERSIFY",
    "REPLACE"

]


strategy_count = (

    strategy_count
    .reindex(
        strategy_order,
        fill_value=0
    )

)


colors = [

    "#2ECC71",
    "#3498DB",
    "#F39C12",
    "#E74C3C"

]


plt.figure(
    figsize=(10,6)
)


bars = plt.bar(

    strategy_count.index,

    strategy_count.values,

    color=colors

)


plt.title(

    "Supplier Management Strategy",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(

    "Number of Suppliers"

)


for bar in bars:

    height = bar.get_height()

    plt.text(

        bar.get_x()
        + bar.get_width()/2,

        height + 0.05,

        int(height),

        ha="center",

        fontweight="bold"

    )


plt.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()

plt.show()


# ================================================================
# 18. IDENTIFICAR PROVEEDORES CRÍTICOS
# ================================================================
#
# Consideraremos críticos:
#
# Risk Score >= 60
#
# O
#
# estrategia = REPLACE
#
# ================================================================


critical_suppliers = df[

    (df["Risk_Score"] >= 60)

    |

    (df["Recommended_Strategy"] == "REPLACE")

]


print("\n" + "="*90)

print("CRITICAL SUPPLIERS REQUIRING IMMEDIATE ATTENTION")

print("="*90)


if len(critical_suppliers) == 0:

    print(

        "No suppliers currently require immediate intervention."

    )

else:

    print(

        critical_suppliers[
            [
                "Supplier",
                "Performance_Score",
                "Risk_Score",
                "Dependency_Percent",
                "Recommended_Strategy"
            ]
        ]

        .round(2)

        .to_string(index=False)

    )


# ================================================================
# 19. INTERPRETACIÓN AUTOMÁTICA
# ================================================================


print("\n" + "="*90)

print("MANAGEMENT RECOMMENDATIONS")

print("="*90)


for i, row in df.iterrows():

    print(

        f"\nSupplier: {row['Supplier']}"

    )

    print(

        f"Performance Score: "
        f"{row['Performance_Score']:.1f}"

    )

    print(

        f"Risk Score: "
        f"{row['Risk_Score']:.1f}"

    )

    print(

        f"Recommended Strategy: "
        f"{row['Recommended_Strategy']}"

    )


    # --------------------------------------------------------
    # Explicación gerencial sencilla.
    # --------------------------------------------------------

    if row["Recommended_Strategy"] == "MAINTAIN":

        print(

            "Interpretation: Strong performance and manageable risk. "
            "Maintain the relationship and continue monitoring."

        )


    elif row["Recommended_Strategy"] == "DEVELOP":

        print(

            "Interpretation: The supplier presents improvement opportunities. "
            "Develop an improvement plan before considering replacement."

        )


    elif row["Recommended_Strategy"] == "DIVERSIFY":

        print(

            "Interpretation: The supplier may perform well, "
            "but dependency or external risk creates vulnerability. "
            "Consider dual sourcing or alternative suppliers."

        )


    elif row["Recommended_Strategy"] == "REPLACE":

        print(

            "Interpretation: Poor performance combined with high risk. "
            "Evaluate replacement or an immediate contingency plan."

        )


# ================================================================
# 20. FINAL MANAGEMENT QUESTIONS
# ================================================================


print("\n" + "="*90)

print("QUESTIONS FOR MANAGEMENT DISCUSSION")

print("="*90)


print("""

1. Which supplier represents the greatest overall risk?

2. Is the supplier with the lowest cost also the best supplier?

3. Which suppliers combine high performance with high risk?

4. Which supplier dependency would concern you the most?

5. Which supplier should receive immediate management attention?

6. Would you replace a risky supplier if it has excellent performance?

7. Which suppliers would you maintain?

8. Which suppliers should be developed?

9. Where would dual sourcing be appropriate?

10. If you were the Supply Chain Manager, what would be your first action?

""")